# 🔥 Fire Detection — YOLOv8 Fine-Tuning on D-Fire Dataset

**Project 10 — Video Analytics for Fire Detection**

| | |
|---|---|
| **Model** | YOLOv8 (fine-tuned from COCO pretrained weights) |
| **Dataset** | D-Fire — 21,527 images, 2 classes (`fire`, `smoke`), YOLO format |
| **Runtime** | Google Colab — **T4 GPU** (Runtime → Change runtime type → T4 GPU) |
| **Expected training time** | ~30–60 min for 100 epochs (yolov8n/s @ 640px) |

**Pipeline:** GPU check → Mount Drive → Unzip dataset → Verify → Train → Evaluate (val) → Final test-set evaluation → Sample predictions → Save everything back to Drive

> ⚠️ **Before running:** upload `dfire_yolov8_ready.zip` (created in Step 1 — local data prep) to your Google Drive, e.g. `MyDrive/fire-detection/dfire_yolov8_ready.zip`

## 1 · GPU sanity check

Confirm Colab actually assigned a T4. If this shows no GPU, go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU** and re-run.

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise SystemError("❌ No GPU detected! Switch runtime to T4 GPU before continuing.")

## 2 · Install dependencies

In [ ]:
%pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 3 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ✏️ EDIT THIS if your zip lives somewhere else in Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/fire-detection"
ZIP_PATH = f"{DRIVE_PROJECT_DIR}/dfire_yolov8_ready.zip"

import os
assert os.path.exists(ZIP_PATH), f"❌ Zip not found at {ZIP_PATH} — upload it to Drive first!"
print(f"✅ Found dataset zip: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e9:.2f} GB)")

## 4 · Unzip dataset to local Colab disk

We extract to `/content/` (fast local SSD) instead of training directly off Drive — Drive I/O is slow and causes training bottlenecks.

In [ ]:
import zipfile
from tqdm.notebook import tqdm

DATA_ROOT = "/content/dfire"
os.makedirs(DATA_ROOT, exist_ok=True)

if os.path.exists(f"{DATA_ROOT}/data/raw/train/images"):
    print("✅ Dataset already extracted — skipping unzip.")
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        members = zf.namelist()
        for m in tqdm(members, desc="Extracting"):
            zf.extract(m, DATA_ROOT)
    print(f"✅ Extracted {len(members)} files to {DATA_ROOT}")

# Show top-level structure
!find {DATA_ROOT} -maxdepth 4 -type d | head -20

## 5 · Verify dataset structure & write Colab-correct `dfire.yaml`

The yaml from the zip contains relative paths from your laptop — we regenerate it with absolute Colab paths to avoid any path confusion.

In [ ]:
import glob, yaml

RAW = f"{DATA_ROOT}/data/raw"

splits = {}
for split in ["train", "val", "test"]:
    n_img = len(glob.glob(f"{RAW}/{split}/images/*"))
    n_lbl = len(glob.glob(f"{RAW}/{split}/labels/*.txt"))
    splits[split] = (n_img, n_lbl)

print(f"{'Split':<8}{'Images':>10}{'Labels':>10}")
print("-" * 28)
total = 0
for s, (ni, nl) in splits.items():
    print(f"{s:<8}{ni:>10}{nl:>10}")
    total += ni
print("-" * 28)
print(f"{'TOTAL':<8}{total:>10}")

assert splits["train"][0] > 0, "❌ train split is empty!"
assert splits["val"][0] > 0,   "❌ val split is empty — did you run 04b_create_val_split.py locally?"
assert splits["test"][0] > 0,  "❌ test split is empty!"

# Quick class distribution scan
from collections import Counter
cls_counts = {s: Counter() for s in splits}
for s in splits:
    for lf in glob.glob(f"{RAW}/{s}/labels/*.txt"):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_counts[s][int(parts[0])] += 1

print(f"\n{'Split':<8}{'fire (0)':>10}{'smoke (1)':>10}")
print("-" * 28)
for s in splits:
    print(f"{s:<8}{cls_counts[s][0]:>10}{cls_counts[s][1]:>10}")

# Write Colab-specific dataset yaml
DATASET_YAML = "/content/dfire.yaml"
cfg = {
    "path": RAW,
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc": 2,
    "names": ["fire", "smoke"],
}
with open(DATASET_YAML, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\n✅ Wrote {DATASET_YAML}:")
print(open(DATASET_YAML).read())

## 6 · Visual sanity check — plot a few labeled samples

Always eyeball a handful of images with their ground-truth boxes drawn before training. Catches axis-swapped or corrupted labels instantly.

In [ ]:
import cv2, random
import matplotlib.pyplot as plt

CLASS_NAMES = {0: "fire", 1: "smoke"}
CLASS_COLORS = {0: (255, 69, 0), 1: (105, 105, 105)}   # RGB: orange-red fire, gray smoke

def draw_sample(img_path):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) == 5:
                    c, cx, cy, bw, bh = int(p[0]), *map(float, p[1:])
                    x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
                    x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), CLASS_COLORS[c], 3)
                    cv2.putText(img, CLASS_NAMES[c], (x1, max(y1 - 8, 15)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, CLASS_COLORS[c], 2)
    return img

train_imgs = glob.glob(f"{RAW}/train/images/*")
random.seed(7)
samples = random.sample(train_imgs, 8)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, p in zip(axes.flat, samples):
    ax.imshow(draw_sample(p))
    ax.set_title(os.path.basename(p), fontsize=8)
    ax.axis("off")
plt.suptitle("Ground-truth samples — D-Fire train split", fontsize=14)
plt.tight_layout()
plt.show()

## 7 · Train YOLOv8

**Model choice for T4 (15 GB VRAM):**

| Model | Params | T4 batch @640 | 100-epoch time | Use when |
|-------|--------|---------------|----------------|----------|
| `yolov8n.pt` | 3.2 M | 64 | ~30 min | fastest iteration, edge deployment |
| **`yolov8s.pt`** ✅ | 11.2 M | 32 | ~50 min | **best accuracy/speed tradeoff — default here** |
| `yolov8m.pt` | 25.9 M | 16 | ~2 hr | max accuracy, if session time allows |

Key training choices:
- `imgsz=640` — D-Fire benchmark standard
- `patience=20` — early-stops if val mAP plateaus for 20 epochs
- `cache=False` — 21k images won't fit Colab RAM; disk read is fine from local SSD
- Augmentation: YOLOv8 defaults (mosaic, HSV jitter, flip) work well for fire/smoke
- Checkpoints are auto-saved every epoch; **cell 9 copies them to Drive** so a session crash loses nothing

In [ ]:
from ultralytics import YOLO

# ✏️ Tweak these if needed
MODEL_SIZE   = "yolov8s.pt"     # yolov8n.pt | yolov8s.pt | yolov8m.pt
EPOCHS       = 100
IMG_SIZE     = 640
BATCH        = 32               # use 64 for v8n, 16 for v8m
PROJECT_NAME = "dfire_yolov8"
RUN_NAME     = "v8s_e100_640"

model = YOLO(MODEL_SIZE)        # downloads COCO-pretrained weights

results = model.train(
    data      = DATASET_YAML,
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH,
    device    = 0,
    project   = PROJECT_NAME,
    name      = RUN_NAME,
    patience  = 20,             # early stopping
    optimizer = "auto",         # AdamW auto-selected with auto-lr
    seed      = 42,
    plots     = True,           # saves loss curves, PR curves, confusion matrix
    val       = True,
    verbose   = True,
)

RUN_DIR = f"{PROJECT_NAME}/{RUN_NAME}"
BEST_PT = f"{RUN_DIR}/weights/best.pt"
print(f"\n✅ Training complete. Best weights: {BEST_PT}")

## 8 · Training curves & validation results

Ultralytics auto-generates all diagnostic plots into the run directory. Display them inline:

In [ ]:
from IPython.display import Image as IPyImage, display

artifacts = [
    ("results.png",            "Loss & metric curves (train/val)"),
    ("confusion_matrix_normalized.png", "Normalized confusion matrix (val)"),
    ("PR_curve.png",           "Precision-Recall curve"),
    ("F1_curve.png",           "F1 vs confidence threshold"),
]

for fname, title in artifacts:
    p = f"{RUN_DIR}/{fname}"
    if os.path.exists(p):
        print(f"\n📊 {title}")
        display(IPyImage(filename=p, width=900))
    else:
        print(f"(missing: {fname})")

### Optimal confidence threshold

The F1 curve above shows which confidence threshold maximizes F1. Note this value — you'll use it as the `conf=` setting in your production inference pipeline (typical for fire models: 0.25–0.40).

## 9 · 💾 Back up run to Google Drive — do this immediately

Colab wipes `/content/` when the session dies. Copy weights + plots to Drive **now**, before anything else.

In [ ]:
import shutil

DRIVE_RUNS = f"{DRIVE_PROJECT_DIR}/runs"
os.makedirs(DRIVE_RUNS, exist_ok=True)

dest = f"{DRIVE_RUNS}/{RUN_NAME}"
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(RUN_DIR, dest)

print(f"✅ Entire run backed up to: {dest}")
print(f"   best.pt → {dest}/weights/best.pt")
print(f"   last.pt → {dest}/weights/last.pt")

## 10 · Final evaluation on the held-out TEST set

This is the number that matters — the model has **never seen** these images during training or checkpoint selection.

In [ ]:
best_model = YOLO(BEST_PT)

test_metrics = best_model.val(
    data  = DATASET_YAML,
    split = "test",          # ← evaluate on test, not val
    imgsz = IMG_SIZE,
    batch = BATCH,
    device= 0,
    plots = True,
    project = PROJECT_NAME,
    name  = f"{RUN_NAME}_test_eval",
)

print("\n" + "=" * 52)
print("📋 FINAL TEST-SET RESULTS")
print("=" * 52)
print(f"  mAP@0.5        : {test_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95   : {test_metrics.box.map:.4f}")
print(f"  Mean Precision : {test_metrics.box.mp:.4f}")
print(f"  Mean Recall    : {test_metrics.box.mr:.4f}")
print("-" * 52)
for i, cname in enumerate(["fire", "smoke"]):
    print(f"  {cname:<6} | AP@0.5: {test_metrics.box.ap50[i]:.4f} | "
          f"AP@0.5:0.95: {test_metrics.box.ap[i]:.4f}")
print("=" * 52)

# Reference: published fine-tuned YOLOv8 results on D-Fire reach ~79-96% mAP@0.5
# depending on model size & training budget. v8s @ 100 epochs typically lands 75-82%.

## 11 · Inference speed benchmark (T4)

Confirms the model meets the real-time CCTV requirement (≥25 FPS).

In [ ]:
import time
import numpy as np

test_imgs = glob.glob(f"{RAW}/test/images/*")[:200]

# Warmup
for p in test_imgs[:10]:
    best_model.predict(p, imgsz=IMG_SIZE, device=0, verbose=False)

t0 = time.time()
for p in test_imgs:
    best_model.predict(p, imgsz=IMG_SIZE, device=0, verbose=False)
elapsed = time.time() - t0

fps = len(test_imgs) / elapsed
print(f"Images processed : {len(test_imgs)}")
print(f"Total time       : {elapsed:.1f} s")
print(f"Throughput       : {fps:.1f} FPS  {'✅ real-time capable' if fps >= 25 else '⚠️ below real-time'}")
print(f"Latency/frame    : {1000/fps:.1f} ms")

## 12 · Visual predictions on test images

Side-by-side check of what the model actually detects — mirrors the `fire 0.41` / `fire 0.66` overlay style from the project slides.

In [ ]:
random.seed(21)
viz_imgs = random.sample(test_imgs, 8)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, p in zip(axes.flat, viz_imgs):
    res = best_model.predict(p, imgsz=IMG_SIZE, conf=0.25, device=0, verbose=False)[0]
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))  # .plot() draws boxes + conf
    n_det = len(res.boxes)
    ax.set_title(f"{os.path.basename(p)} — {n_det} detection(s)", fontsize=8)
    ax.axis("off")
plt.suptitle("YOLOv8 predictions on held-out test images (conf ≥ 0.25)", fontsize=14)
plt.tight_layout()
plt.show()

## 13 · (Optional) Video inference demo

Upload any fire video clip (e.g. `demo_fire.mp4`) to `MyDrive/fire-detection/` and run this cell to produce an annotated output video — exactly the bounding-box-overlay output shown in the project slides.

In [ ]:
VIDEO_IN = f"{DRIVE_PROJECT_DIR}/demo_fire.mp4"   # ✏️ your test clip

if os.path.exists(VIDEO_IN):
    results = best_model.predict(
        source  = VIDEO_IN,
        imgsz   = IMG_SIZE,
        conf    = 0.25,
        device  = 0,
        save    = True,                 # writes annotated .mp4
        project = PROJECT_NAME,
        name    = f"{RUN_NAME}_video_demo",
        verbose = False,
    )
    out_dir = f"{PROJECT_NAME}/{RUN_NAME}_video_demo"
    print(f"✅ Annotated video saved in: {out_dir}")
    # Copy to Drive
    for f in glob.glob(f"{out_dir}/*.mp4") + glob.glob(f"{out_dir}/*.avi"):
        shutil.copy(f, DRIVE_PROJECT_DIR)
        print(f"   → copied to Drive: {os.path.basename(f)}")
else:
    print(f"ℹ️ No video found at {VIDEO_IN} — skip or upload a clip to run this demo.")

## 14 · Export model for deployment + final Drive sync

Exports ONNX (universal CPU/GPU deployment) — usable in your local inference pipeline (Step 5 of the workflow) without needing the full Ultralytics training stack.

In [ ]:
# Export to ONNX for the local inference pipeline
onnx_path = best_model.export(format="onnx", imgsz=IMG_SIZE, simplify=True)
print(f"✅ ONNX export: {onnx_path}")

# Sync test-eval artifacts + ONNX to Drive
test_eval_dir = f"{PROJECT_NAME}/{RUN_NAME}_test_eval"
if os.path.exists(test_eval_dir):
    dest = f"{DRIVE_RUNS}/{RUN_NAME}_test_eval"
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(test_eval_dir, dest)
    print(f"✅ Test eval artifacts → {dest}")

if os.path.exists(str(onnx_path)):
    shutil.copy(str(onnx_path), f"{DRIVE_RUNS}/{RUN_NAME}/weights/")
    print(f"✅ ONNX model → {DRIVE_RUNS}/{RUN_NAME}/weights/")

print("\n🎉 All done! Final deliverables in Drive:")
print(f"   {DRIVE_RUNS}/{RUN_NAME}/weights/best.pt   ← PyTorch weights")
print(f"   {DRIVE_RUNS}/{RUN_NAME}/weights/best.onnx ← deployment model")
print(f"   {DRIVE_RUNS}/{RUN_NAME}/                  ← all plots & curves")

---
## ✅ Summary & next steps

**What this notebook produced:**
1. Fine-tuned YOLOv8s on D-Fire (fire + smoke, 2 classes)
2. Validation curves, confusion matrix, PR/F1 curves
3. Unbiased test-set metrics (mAP@0.5, mAP@0.5:0.95, per-class AP)
4. T4 inference speed benchmark
5. `best.pt` + `best.onnx` saved to Google Drive

**Next steps (Step 5 — local inference):**
- Download `best.pt` / `best.onnx` from Drive to your laptop
- Build the real-time CCTV inference script (OpenCV `VideoCapture` → YOLO → bbox overlay → alert trigger)
- Add the alert-dispatch logic from the project spec (confidence threshold + persistence filter to suppress false alarms)

**If accuracy needs a boost:**
- Train `yolov8m` (longer but +2-4% mAP)
- Increase epochs to 150-200 with `patience=30`
- Add the Roboflow METU indoor/industrial dataset for CCTV-specific coverage
- Tune `conf` threshold per the F1 curve from cell 8